# Day 1 · 03 · Power BI Data Preparation and Semantic Model Handoff

![Power BI handoff map](../../assets/images/day1_03_powerbi_handoff_map.png)

Workshop 1 built the Gold model. This notebook validates that model for Power BI, explains semantic-model decisions and prepares a practical handoff checklist.


## Business Scenario

The BI team now has Gold objects in Databricks. The next decision is not whether Power BI can connect, but what it should connect to, which logic belongs in Databricks, and which logic should remain in the Power BI semantic model.


## Learning Objectives

By the end of this notebook you can:

- validate the Workshop 1 Kimball model,
- confirm grain and relationship readiness,
- choose between a star schema and a flat dashboard table,
- decide what belongs in Databricks Gold vs Power BI DAX or Power Query,
- define baseline KPI measures,
- explain Import, DirectQuery and Composite model tradeoffs,
- prepare a handoff checklist for the Power BI team.


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `silver.order_lines`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before running SQL cells.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

Workshop 1 must have created the complete Gold model. If this fails, run `w1_gold_kpi_solution.ipynb` first.


In [ ]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Complete Workshop 1 before running Day1_03.")
print("[OK] Workshop 1 Gold model is available.")


## 1. Gold Inventory

Start with a concrete inventory. A Power BI handoff should name the catalog, schema, objects and expected row counts.


### Definition: Power BI Semantic Model

A Power BI semantic model is the governed layer inside Power BI that contains tables, relationships, measures, hierarchies, formatting and refresh mode. It is not just a report file; it is the contract that report pages query.

Expected observation: Databricks Gold provides trusted tables, while the semantic model decides how business users navigate and calculate over those tables.


In [ ]:
%sql
SHOW TABLES IN gold


In [ ]:
%sql
SELECT 'dim_date' AS object_name, COUNT(*) AS rows FROM gold.dim_date
UNION ALL
SELECT 'dim_customer', COUNT(*) FROM gold.dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*) FROM gold.dim_product
UNION ALL
SELECT 'fact_sales', COUNT(*) FROM gold.fact_sales
UNION ALL
SELECT 'fact_sales_dashboard', COUNT(*) FROM gold.fact_sales_dashboard
ORDER BY object_name


## 2. Grain Checks

Power BI visuals break when a table's grain is unclear. The fact and dashboard table should both have one row per `line_id`.


### Definition: Grain Check

A grain check proves what one row represents. In this model, `fact_sales` and `fact_sales_dashboard` should both have one row per `line_id`.

Pitfall: if a dashboard table has more rows than the fact table, a Power BI measure can overcount revenue even when the DAX formula looks correct.


In [ ]:
%sql
SELECT
  'fact_sales' AS object_name,
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_lines,
  COUNT(DISTINCT order_id) AS distinct_orders
FROM gold.fact_sales
UNION ALL
SELECT
  'fact_sales_dashboard',
  COUNT(*),
  COUNT(DISTINCT line_id),
  COUNT(*) - COUNT(DISTINCT line_id),
  COUNT(DISTINCT order_id)
FROM gold.fact_sales_dashboard


## 3. Relationship Readiness

A semantic model relationship needs a unique dimension key and matching fact rows. Check both before connecting Power BI.


### Definition: Relationship Readiness and Orphan Check

Relationship readiness means that dimension keys are unique and fact rows can find matching dimension rows. Power BI many-to-one relationships depend on both conditions.

An orphan check finds fact rows with no matching dimension row. Orphans create blanks, unexpected slicer behavior or row loss if someone later changes a left join to an inner join.


In [ ]:
%sql
SELECT 'dim_date' AS dimension, COUNT(*) AS rows, COUNT(DISTINCT date_key) AS distinct_keys, COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys
FROM gold.dim_date
UNION ALL
SELECT 'dim_customer', COUNT(*), COUNT(DISTINCT customer_id), COUNT(*) - COUNT(DISTINCT customer_id)
FROM gold.dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*), COUNT(DISTINCT product_id), COUNT(*) - COUNT(DISTINCT product_id)
FROM gold.dim_product


In [ ]:
%sql
SELECT 'fact_sales.order_date -> dim_date.date_key' AS relationship, COUNT(*) AS orphan_rows
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_date d ON f.order_date = d.date_key
UNION ALL
SELECT 'fact_sales.customer_id -> dim_customer.customer_id', COUNT(*)
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_customer c ON f.customer_id = c.customer_id
UNION ALL
SELECT 'fact_sales.product_id -> dim_product.product_id', COUNT(*)
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_product p ON f.product_id = p.product_id


## 4. Star Schema vs Flat Table

![Star schema vs flat table](../../assets/images/star_schema_vs_flat_table.png)

Use the star schema for governed semantic models. Use the flat dashboard table when the training goal is speed, first exploration or a narrow operational report.

| Source option | Best for | Tradeoff |
|---|---|---|
| `gold.fact_sales` + dimensions | reusable semantic model, relationships, governed measures | more modeling work in Power BI |
| `gold.fact_sales_dashboard` | quick report pages and prototypes | duplicated attributes and wider DirectQuery scans |

Expected observation: both options are valid, but they solve different delivery problems.


### Relationship Map

| From table | From column | To table | To column | Cardinality | Cross-filter direction |
|---|---|---|---|---|---|
| `gold.fact_sales` | `order_date` | `gold.dim_date` | `date_key` | many-to-one | single |
| `gold.fact_sales` | `customer_id` | `gold.dim_customer` | `customer_id` | many-to-one | single |
| `gold.fact_sales` | `product_id` | `gold.dim_product` | `product_id` | many-to-one | single |


### Dashboard vs Fact Reconciliation

The wide dashboard table should reconcile to the governed fact table. This check confirms that the dashboard table did not duplicate or drop fact rows.


In [ ]:
%sql
WITH fact AS (
  SELECT
    COUNT(*) AS fact_rows,
    COUNT(DISTINCT line_id) AS fact_lines,
    ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS fact_revenue
  FROM gold.fact_sales
),
dashboard AS (
  SELECT
    COUNT(*) AS dashboard_rows,
    COUNT(DISTINCT line_id) AS dashboard_lines,
    ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS dashboard_revenue
  FROM gold.fact_sales_dashboard
)
SELECT
  fact_rows,
  dashboard_rows,
  fact_rows - dashboard_rows AS row_diff,
  fact_lines,
  dashboard_lines,
  fact_lines - dashboard_lines AS line_diff,
  fact_revenue,
  dashboard_revenue,
  ROUND(fact_revenue - dashboard_revenue, 2) AS revenue_diff
FROM fact
CROSS JOIN dashboard


Expected observation: row, line and revenue differences should be zero. If not, inspect joins in `gold.fact_sales_dashboard` before connecting Power BI.


## 5. Databricks Gold vs Power BI Logic

Keep shared, expensive and governed logic in Databricks. Keep interactive report semantics in Power BI.

| Logic | Prefer Databricks Gold | Prefer Power BI |
|---|---|---|
| joins and conformed attributes | yes | no |
| row-level cleansing and unknown labels | yes | no |
| reusable KPI base columns | yes | sometimes |
| report-specific slicer labels | sometimes | yes |
| visual-specific measures | no | yes |
| ad hoc formatting and presentation | no | yes |
| Power Query transformations over large DirectQuery tables | no | avoid |


### DAX vs Databricks SQL Decision Table

| Logic | Best home | Reason |
|---|---|---|
| joins to conformed dimensions | Databricks Gold | shared, expensive and reusable |
| unknown labels and source quality flags | Databricks Gold | consistent across all BI tools |
| additive base measures | Databricks Gold fields | easy to validate and reuse |
| report-specific measures | Power BI DAX | interactive and presentation-aware |
| time intelligence formatting | Power BI DAX | depends on report calendar behavior |
| large row-by-row transformations | Databricks SQL | avoid Power Query work over DirectQuery |
| column renames for report readability | Power BI or Gold | Gold for shared terms, Power BI for report-only labels |

Expected observation: the decision is about ownership. Shared logic belongs closer to Gold; visual logic belongs in the semantic model.


## 6. KPI Dictionary

KPI definitions must be explicit before report authors build visuals. Ambiguous filters are the fastest way to create inconsistent dashboards.


### Definition: DAX Measure

A DAX measure is a reusable calculation evaluated at query time in the current filter context. Measures should express business logic such as revenue, margin rate or return rate.

Pitfall: a measure can be correct syntactically but wrong semantically if the underlying table grain or relationships are wrong.


In [ ]:
%sql
SELECT
  'Revenue' AS kpi_name,
  'SUM(line_revenue) for completed rows' AS definition,
  'Currency' AS format,
  'gold.fact_sales or gold.fact_sales_dashboard' AS source_object
UNION ALL
SELECT 'Gross Margin', 'SUM(line_margin) for completed rows', 'Currency', 'gold.fact_sales or gold.fact_sales_dashboard'
UNION ALL
SELECT 'Completed Orders', 'DISTINCTCOUNT(order_id) for completed rows', 'Whole number', 'gold.fact_sales or gold.fact_sales_dashboard'
UNION ALL
SELECT 'Margin Rate %', 'Gross Margin divided by Revenue', 'Percentage', 'Power BI measure over Gold fields'
UNION ALL
SELECT 'Return Rate %', 'Returned orders divided by completed + returned orders', 'Percentage', 'Power BI measure over Gold fields'


In [ ]:
%sql
SELECT
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  ROUND(
    100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END)
    / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0),
    2
  ) AS margin_rate_pct
FROM gold.fact_sales_dashboard


### Starter DAX Measures

```DAX
Revenue =
CALCULATE (
    SUM ( fact_sales[line_revenue] ),
    fact_sales[is_completed] = TRUE ()
)

Gross Margin =
CALCULATE (
    SUM ( fact_sales[line_margin] ),
    fact_sales[is_completed] = TRUE ()
)

Completed Orders =
CALCULATE (
    DISTINCTCOUNT ( fact_sales[order_id] ),
    fact_sales[is_completed] = TRUE ()
)

Margin Rate % =
DIVIDE ( [Gross Margin], [Revenue] )
```

Expected observation: Databricks supplies governed fields and filters; DAX defines reusable report measures.


## 7. Prepared Preview for Power BI

This temporary view previews a monthly report page without creating another durable Gold object.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW powerbi_monthly_preview AS
SELECT
  year_month,
  customer_region,
  category,
  channel,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders
FROM gold.fact_sales_dashboard
GROUP BY year_month, customer_region, category, channel


In [ ]:
%sql
SELECT *
FROM powerbi_monthly_preview
ORDER BY year_month DESC, revenue DESC
LIMIT 20


## 8. Freshness and Row Count Trend Checks

Freshness tells report owners how current the Gold model is. A row count trend gives a lightweight sanity check before scheduled refresh.


In [ ]:
%sql
SELECT
  MAX(order_date) AS max_order_date,
  current_date() AS checked_on,
  datediff(current_date(), MAX(order_date)) AS days_since_latest_order
FROM gold.fact_sales_dashboard


In [ ]:
%sql
SELECT
  year_month,
  COUNT(*) AS rows,
  COUNT(DISTINCT order_id) AS orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS completed_revenue
FROM gold.fact_sales_dashboard
GROUP BY year_month
ORDER BY year_month DESC
LIMIT 12


Expected observation: freshness is a business agreement, not only a technical timestamp. The trend check helps detect missing or duplicated monthly loads before Power BI refresh.


## 9. Import vs DirectQuery vs Composite

![Import vs DirectQuery](../../assets/images/import_vs_directquery.png)

`Import` copies data into the Power BI model. Choose it for fast executive dashboards, stable refresh windows and high interactivity.

`DirectQuery` sends user interactions back to the SQL Warehouse. Choose it only when freshness requirements justify the extra latency, governance and cost sensitivity.

`Composite` mixes Import and DirectQuery tables. Choose it when some tables need live behavior but most dimensions or aggregates can be imported.

Pitfall: DirectQuery over a wide line-grain table can multiply warehouse queries through slicers, visuals and cross-filtering.

| Mode | When to choose | Watch out |
|---|---|---|
| Import | scheduled BI, high performance, many users | refresh window and dataset size |
| DirectQuery | live operational view, controlled visuals | query fan-out and SQL Warehouse cost |
| Composite | mixed freshness needs | model complexity and relationship behavior |

Expected observation: the default recommendation for this course is Import unless the business explicitly requires live query behavior.


### Query Folding and Incremental Refresh Awareness

Power Query steps should fold back to Databricks SQL. Filters on dates and simple projections usually fold; custom row-by-row transformations often do not. For Import models, incremental refresh should filter on a stable date column such as `order_date`.


### Definitions: Query Folding and Incremental Refresh

Query folding means Power Query can translate transformations back into source SQL. When folding works, Databricks does the heavy work. When folding breaks, Power BI may do more work locally or generate inefficient source queries.

Incremental refresh loads only new or changed date ranges instead of refreshing the full dataset. It requires a stable date column such as `order_date` and a clear refresh policy.

Pitfall: complex Power Query transformations can break folding. Prefer shaping large data in Databricks Gold before Power BI connects.


In [ ]:
%sql
EXPLAIN
SELECT
  year_month,
  channel,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard
WHERE order_date >= DATE '2024-01-01'
GROUP BY year_month, channel


## 10. Minimal Power BI Semantic Model

Recommended starting model for the governed path:

| Table | Role | Hide from report view? |
|---|---|---|
| `gold.fact_sales` | central fact table | no |
| `gold.dim_date` | date dimension | no |
| `gold.dim_customer` | customer slicers | no |
| `gold.dim_product` | product slicers | no |
| technical keys such as `line_id` | relationship/debug fields | usually yes |

Recommended slicers: `dim_date[year_month]`, `dim_customer[customer_region]`, `dim_customer[segment]`, `dim_product[category]`, `fact_sales[channel]`.

Recommended measures: Revenue, Gross Margin, Completed Orders, Margin Rate %, Return Rate %.

Expected observation: a semantic model should expose business fields and hide technical clutter without removing the governed keys needed for relationships.


## 11. Power BI Handoff Packet

A professional handoff is a packet of decisions, not just a connection string.

| Handoff item | Required decision |
|---|---|
| Connection details | SQL Warehouse hostname and HTTP path from Databricks UI |
| Authentication | PAT, OAuth or Entra ID pattern chosen by workspace policy |
| Source option | star schema for governed model, dashboard table for quick prototype |
| Relationship map | fact date/customer/product keys to unique dimensions |
| Model mode | Import by default, DirectQuery only with explicit freshness need |
| Refresh expectation | schedule, owner and freshness SLA |
| Cost guardrails | warehouse size, auto-stop, visual count and DirectQuery limits |
| Measure ownership | which measures are DAX-owned vs Gold-owned |

Expected observation: this table is the conversation checklist between Databricks and Power BI owners.


## 12. Connector Handoff Checklist

| Item | Value or source |
|---|---|
| Catalog | value from the configuration cell |
| Schema | `gold` |
| SQL Warehouse hostname | SQL Warehouses -> Connection details |
| HTTP path | SQL Warehouses -> Connection details |
| Authentication | PAT token or Entra ID, depending on workspace setup |
| First table for quick report | `gold.fact_sales_dashboard` |
| Star schema option | `gold.fact_sales`, `gold.dim_date`, `gold.dim_customer`, `gold.dim_product` |
| Relationships | fact date/customer/product keys to dimensions |
| Recommended measures | revenue, gross margin, completed orders, margin rate, return rate |
| Refresh expectation | align Power BI refresh with Gold refresh job |
| Cost guardrail | avoid wide DirectQuery visuals with many slicers unless latency requires DirectQuery |

Trainer note: during delivery, open SQL Warehouse connection details in the Databricks UI and point out the hostname and HTTP path that Power BI requires.


## Trainer Delivery Notes

- Ask participants to identify the grain before they read each KPI query.
- Emphasize that `fact_sales_dashboard` is convenient, while the star schema is the governed model.
- Use the reconciliation query before showing DAX measures; it proves that the semantic model starts from trustworthy rows.
- Pause at Import vs DirectQuery and ask which mode they would choose for executive dashboards.
- Treat the handoff packet as the final Day 1 artifact: it connects Databricks engineering work with Power BI delivery.


## Summary and Day 2 Handoff

Day 1 now has a complete path: platform foundation, SQL and architecture patterns, full Kimball workshop and Power BI readiness. Day 2 can focus on performance, dashboards, automation and production handoff instead of rebuilding the model again.
